# RAID Multi-Genre Analysis: Memory Curves Across Domains & Models

**Question:** Does the memory curve shape invariance (human vs AI) hold across genres and AI models?

**Design:**
- 8 genres: abstracts, books, news, poetry, recipes, reddit, reviews, wiki
- 6 sources: human + 5 AI models (ChatGPT, GPT-4, Llama, Mistral, Cohere)
- Shorter window config (texts are ~200-400 words)
- Test: is curve shape invariant to authorship across genres?

In [ ]:
# ── Install & Import ──
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
import statsmodels.formula.api as smf
from pathlib import Path
import json, math, time, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
# ── Config ──
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/LRTIA/Results/RAID")
    DATA_DIR = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    IN_COLAB = True
except:
    BASE_DIR = Path("../results/raid")
    DATA_DIR = Path("../data/raid_sampled")
    IN_COLAB = False

BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

# Shorter windows for shorter texts — dense at the early end
WINDOWS = [4, 8, 12, 16, 24, 32, 48, 64, 96, 128]
BURN_IN = 128
MAX_SCORE_TOKENS = 64  # Short scoring region for short texts
BUFFER = 10
MIN_TOKENS = BURN_IN + MAX_SCORE_TOKENS + BUFFER  # 202

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']
COLORS_POP = {'human': '#3498db', 'ai': '#e74c3c'}
COLORS_DOMAIN = dict(zip(DOMAINS, plt.cm.Set2.colors[:len(DOMAINS)]))

print(f"Windows: {WINDOWS}")
print(f"Min tokens: {MIN_TOKENS}")

In [ ]:
# ── Load corpus ──
corpus_path = DATA_DIR / "raid_corpus.jsonl"
corpus = []
with open(corpus_path) as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus)} documents")
print(f"\nBy domain x population:")
for d in DOMAINS:
    nh = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'human')
    na = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'ai')
    print(f"  {d:<15} human={nh}, ai={na}")

print(f"\nAI models: {sorted(set(c['model'] for c in corpus if c['population'] == 'ai'))}")

In [ ]:
# ── Load model ──
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")

model.eval()
print("Model loaded")

In [ ]:
# ── Core functions ──
@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return float('inf'), 0
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    return math.exp(total_loss / count) if count > 0 else float('inf'), count

def compute_memory_curve(token_ids, windows, burn_in, max_score_tokens):
    result = {'ppl_by_W': {}, 'token_count': len(token_ids)}
    n_tokens = len(token_ids)
    if n_tokens <= burn_in:
        return result
    target_end = min(n_tokens, burn_in + max_score_tokens)
    for W in windows:
        context_start = max(0, burn_in - W)
        actual_context = burn_in - context_start
        if actual_context < 4:
            continue
        truncated = token_ids[context_start:target_end]
        ppl, _ = compute_perplexity_on_region(truncated, actual_context, len(truncated))
        if not math.isinf(ppl):
            result['ppl_by_W'][W] = ppl
    return result

def compute_half_life(ppl_dict, percentile=0.5):
    if len(ppl_dict) < 2:
        return float('nan')
    items = sorted(ppl_dict.items())
    windows = np.array([x[0] for x in items])
    ppls = np.array([x[1] for x in items])
    total_benefit = ppls[0] - ppls[-1]
    if total_benefit <= 0:
        return float('nan')
    target_ppl = ppls[0] - percentile * total_benefit
    for i in range(len(ppls) - 1):
        if ppls[i] >= target_ppl >= ppls[i + 1]:
            frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i + 1])
            return windows[i] + frac * (windows[i + 1] - windows[i])
    return windows[-1]

print("Functions defined")

In [ ]:
# ── Compute memory curves ──
results = []
skipped = 0

for doc in tqdm(corpus, desc="Computing memory curves"):
    token_ids = tokenizer.encode(doc["text"], add_special_tokens=False)
    n_tokens = len(token_ids)

    if n_tokens < MIN_TOKENS:
        skipped += 1
        continue

    curve = compute_memory_curve(token_ids, WINDOWS, BURN_IN, MAX_SCORE_TOKENS)
    if len(curve['ppl_by_W']) < 3:
        skipped += 1
        continue

    ppl_by_W = curve['ppl_by_W']
    computed_W = sorted(ppl_by_W.keys())
    computed_ppls = np.array([ppl_by_W[w] for w in computed_W])
    delta_ppl = computed_ppls[0] - computed_ppls

    row = {
        'doc_id': doc['doc_id'],
        'domain': doc['domain'],
        'model': doc['model'],
        'population': doc['population'],
        'token_count': n_tokens,
        'half_life': compute_half_life(ppl_by_W),
    }

    for W in WINDOWS:
        row[f'ppl_W{W}'] = ppl_by_W.get(W, np.nan)

    row['delta_max'] = delta_ppl[-1]
    row['auc_full'] = trapezoid(delta_ppl, computed_W)

    if len(computed_W) >= 3:
        slope, _, _, _, _ = stats.linregress(np.log(computed_W), delta_ppl)
        row['log_slope'] = slope

    # Early (<=32) vs late (>=32) AUC
    early_W = [w for w in computed_W if w <= 32]
    late_W = [w for w in computed_W if w >= 32]
    if len(early_W) >= 2:
        ep = np.array([ppl_by_W[w] for w in early_W])
        row['auc_early'] = trapezoid(ep[0] - ep, early_W)
    if len(late_W) >= 2:
        lp = np.array([ppl_by_W[w] for w in late_W])
        row['auc_late'] = trapezoid(lp[0] - lp, late_W)
    if row.get('auc_early') and row.get('auc_full') and row['auc_full'] > 0:
        row['early_fraction'] = row['auc_early'] / row['auc_full']

    results.append(row)

df = pd.DataFrame(results)
print(f"\nProcessed {len(df)} docs ({skipped} skipped)")
print(f"  Human: {len(df[df.population == 'human'])}")
print(f"  AI:    {len(df[df.population == 'ai'])}")

df.to_csv(BASE_DIR / 'essay_level_results.csv', index=False)
print(f"Saved to {BASE_DIR / 'essay_level_results.csv'}")

## Key Question: Does Curve Shape Differ by Genre or Authorship?

In [ ]:
# ── Figure 1: Normalized curves by domain (human vs AI overlay) ──
ppl_cols = [f'ppl_W{w}' for w in WINDOWS if f'ppl_W{w}' in df.columns]
plot_W = [int(c.replace('ppl_W', '')) for c in ppl_cols]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, domain in enumerate(DOMAINS):
    ax = axes[idx // 4, idx % 4]
    for pop, ls, lw in [('human', '-', 2.5), ('ai', '--', 1.5)]:
        sub = df[(df.domain == domain) & (df.population == pop)]
        if len(sub) == 0:
            continue
        means = np.array([sub[c].mean() for c in ppl_cols])
        total_drop = means[0] - means[-1]
        if total_drop > 0:
            normalized = (means[0] - means) / total_drop
        else:
            normalized = np.zeros_like(means)
        ax.plot(plot_W, normalized, marker='o', linestyle=ls, linewidth=lw,
                color=COLORS_POP[pop], markersize=4, label=pop)

    ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(domain, fontsize=12, fontweight='bold')
    ax.set_xscale('log', base=2)
    ax.set_ylim(-0.05, 1.05)
    if idx % 4 == 0:
        ax.set_ylabel('Fraction of Total Benefit')
    if idx >= 4:
        ax.set_xlabel('Context Window')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=8)

plt.suptitle('Normalized Memory Curves by Genre: Human (solid) vs AI (dashed)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_curves_by_domain.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Half-life and early_fraction by domain x population ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (metric, label) in enumerate([
    ('half_life', 'Half-Life (tokens)'),
    ('early_fraction', 'Early Fraction (AUC early/full)'),
    ('ppl_W4', 'Baseline Perplexity (W4)')
]):
    ax = axes[idx]
    if metric not in df.columns:
        continue

    data_h = [df[(df.domain == d) & (df.population == 'human')][metric].dropna().values for d in DOMAINS]
    data_a = [df[(df.domain == d) & (df.population == 'ai')][metric].dropna().values for d in DOMAINS]

    x = np.arange(len(DOMAINS))
    w = 0.35
    h_means = [np.mean(v) if len(v) > 0 else 0 for v in data_h]
    a_means = [np.mean(v) if len(v) > 0 else 0 for v in data_a]
    h_sems = [stats.sem(v) if len(v) > 1 else 0 for v in data_h]
    a_sems = [stats.sem(v) if len(v) > 1 else 0 for v in data_a]

    ax.bar(x - w/2, h_means, w, yerr=h_sems, label='Human',
           color=COLORS_POP['human'], alpha=0.8, capsize=2)
    ax.bar(x + w/2, a_means, w, yerr=a_sems, label='AI',
           color=COLORS_POP['ai'], alpha=0.8, capsize=2)

    ax.set_xticks(x)
    ax.set_xticklabels(DOMAINS, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_metrics_by_domain.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: Curves by AI model (within a single domain) ──
# Pick news as representative — longest texts, most natural comparison
focus_domain = 'news'
ai_models = sorted(df[df.population == 'ai']['model'].unique())
model_colors = dict(zip(ai_models, plt.cm.tab10.colors[:len(ai_models)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3a: Raw curves
ax = axes[0]
sub_h = df[(df.domain == focus_domain) & (df.population == 'human')]
means_h = np.array([sub_h[c].mean() for c in ppl_cols])
ax.plot(plot_W, means_h, 'k-o', linewidth=3, markersize=6, label='human', zorder=10)

for m in ai_models:
    sub = df[(df.domain == focus_domain) & (df.model == m)]
    if len(sub) == 0:
        continue
    means = np.array([sub[c].mean() for c in ppl_cols])
    ax.plot(plot_W, means, '--o', color=model_colors[m], linewidth=1.5, markersize=4, label=m)

ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window')
ax.set_ylabel('Perplexity')
ax.set_title(f'Raw Curves: {focus_domain} (by model)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3b: Normalized curves
ax = axes[1]
total_h = means_h[0] - means_h[-1]
if total_h > 0:
    ax.plot(plot_W, (means_h[0] - means_h) / total_h, 'k-o', linewidth=3, markersize=6, label='human', zorder=10)

for m in ai_models:
    sub = df[(df.domain == focus_domain) & (df.model == m)]
    if len(sub) == 0:
        continue
    means = np.array([sub[c].mean() for c in ppl_cols])
    total = means[0] - means[-1]
    if total > 0:
        ax.plot(plot_W, (means[0] - means) / total, '--o', color=model_colors[m],
                linewidth=1.5, markersize=4, label=m)

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xscale('log', base=2)
ax.set_xlabel('Context Window')
ax.set_ylabel('Fraction of Total Benefit')
ax.set_title(f'Normalized Curves: {focus_domain} (by model)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'fig3_by_model.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Tests

In [ ]:
# ── Per-domain human vs AI tests ──
print("="*80)
print("SHAPE INVARIANCE TEST: early_fraction & half_life by domain")
print("="*80)

shape_metrics = ['half_life', 'early_fraction']
shape_metrics = [m for m in shape_metrics if m in df.columns]

for metric in shape_metrics:
    print(f"\n--- {metric} ---")
    print(f"{'Domain':<15} {'Human':>10} {'AI':>10} {'d':>8} {'p':>10} {'Sig':>5}")
    print("-"*58)
    for d in DOMAINS:
        h = df[(df.domain == d) & (df.population == 'human')][metric].dropna()
        a = df[(df.domain == d) & (df.population == 'ai')][metric].dropna()
        if len(h) < 3 or len(a) < 3:
            print(f"{d:<15} {'insufficient data':>40}")
            continue
        t, p = stats.ttest_ind(h, a)
        cohen_d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{d:<15} {h.mean():>10.3f} {a.mean():>10.3f} {cohen_d:>8.3f} {p:>9.4f} {sig:>5}")

# Overall 2-way ANOVA: domain x population on shape metrics
print("\n\n" + "="*80)
print("2-WAY ANOVA: domain x population")
print("="*80)

df['is_ai'] = (df['population'] == 'ai').astype(int)
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()

for metric in shape_metrics:
    if metric not in df.columns:
        continue
    formula = f'{metric} ~ C(domain) * is_ai + token_count_z'
    try:
        m = smf.ols(formula, data=df.dropna(subset=[metric])).fit()
        print(f"\n{metric}: R2={m.rsquared:.4f}")
        # Check interaction terms
        interaction_terms = [p for p in m.params.index if ':' in p]
        if interaction_terms:
            print("  Interaction terms (domain x is_ai):")
            for term in interaction_terms:
                p = m.pvalues[term]
                sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
                print(f"    {term}: beta={m.params[term]:+.4f}, p={p:.4f} {sig}")

        # Main effect of is_ai
        p_ai = m.pvalues.get('is_ai', 1)
        b_ai = m.params.get('is_ai', 0)
        sig = "***" if p_ai < 0.001 else "**" if p_ai < 0.01 else "*" if p_ai < 0.05 else "ns"
        print(f"  Main effect of AI: beta={b_ai:+.4f}, p={p_ai:.4f} {sig}")
    except Exception as e:
        print(f"  ERROR: {e}")

In [ ]:
# ── Summary ──
print("="*70)
print("SUMMARY: Multi-Genre Memory Curve Analysis")
print("="*70)

print(f"\nDataset: RAID ({len(df)} docs)")
print(f"Domains: {len(df.domain.unique())}")
print(f"Models: {sorted(df.model.unique())}")
print(f"Windows: {WINDOWS}")

print(f"\nDoes genre affect curve shape?")
for m in ['half_life', 'early_fraction']:
    if m not in df.columns:
        continue
    f, p = stats.f_oneway(*[df[df.domain == d][m].dropna() for d in DOMAINS if len(df[df.domain == d][m].dropna()) > 2])
    sig = "YES" if p < 0.05 else "NO"
    print(f"  {m}: F={f:.2f}, p={p:.4f} -> {sig}")

print(f"\nDoes human vs AI affect curve shape (controlling for domain)?")
for m in ['half_life', 'early_fraction']:
    if m not in df.columns:
        continue
    h = df[df.population == 'human'][m].dropna()
    a = df[df.population == 'ai'][m].dropna()
    t, p = stats.ttest_ind(h, a)
    d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
    sig = "YES" if p < 0.05 else "NO"
    print(f"  {m}: d={d:.3f}, p={p:.4f} -> {sig}")

print(f"\nDoes human vs AI affect baseline perplexity?")
h = df[df.population == 'human']['ppl_W4'].dropna()
a = df[df.population == 'ai']['ppl_W4'].dropna()
t, p = stats.ttest_ind(h, a)
d = (h.mean() - a.mean()) / np.sqrt((h.std()**2 + a.std()**2) / 2)
print(f"  ppl_W4: human={h.mean():.1f}, ai={a.mean():.1f}, d={d:.3f}, p={p:.4f}")

df.to_csv(BASE_DIR / 'essay_level_results.csv', index=False)
print(f"\nAll results saved to {BASE_DIR}/")